# 📊 Coletor de Dataset de Gestos
**Baseado em:** `reconhecimento_gestos_webcam.ipynb`

Grava **continuamente** as posições dos 21 landmarks de cada mão e salva em CSV.

**Fluxo de uso:**
1. Execute o loop — ele pedirá o nome do gesto inicial no terminal
2. Faça o gesto na frente da câmera — gravação automática a cada frame
3. Para trocar o gesto: **digite o novo nome no terminal e pressione ENTER**
4. Pressione **Q** na janela da câmera para encerrar

## 📦 1. Instalação das dependências
```bash
uv add opencv-python mediapipe pandas
```

In [1]:
# Descomente se precisar instalar no ambiente do notebook
# !pip install opencv-python mediapipe pandas -q

## 📚 2. Importações

In [2]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
import numpy as np
import pandas as pd
import os

## 📥 3. Download do modelo

In [3]:
import urllib.request

MODEL_URL  = "https://storage.googleapis.com/mediapipe-models/gesture_recognizer/gesture_recognizer/float16/1/gesture_recognizer.task"
MODEL_PATH = "gesture_recognizer.task"

if not os.path.exists(MODEL_PATH):
    print("Baixando modelo...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print(f"✅ Modelo salvo em: {MODEL_PATH}")
else:
    print(f"✅ Modelo já existe: {MODEL_PATH}")

✅ Modelo já existe: gesture_recognizer.task


## ⚙️ 4. Configurações

> **Edite `LABEL` antes de cada sessão de gravação.**  
> Exemplo: `"thumbs_up"`, `"victory"`, `"fist"`, etc.

In [4]:
# ── Arquivo CSV de saída ──────────────────────────────────────────────────────
#    Se o arquivo já existir, os novos dados são ADICIONADOS (append)
CSV_PATH = "dataset_gestos.csv"

# ── Configurações da câmera e modelo ─────────────────────────────────────────
CAMERA_INDEX    = 0
SCORE_THRESHOLD = 0.5
MAX_HANDS       = 1   # 1 mão por frame para o dataset ficar consistente

# ── Cores (BGR) ───────────────────────────────────────────────────────────────
COR_GRAVANDO = (0, 255, 120)    # verde — gravando
COR_ERRO     = (0, 80, 255)     # laranja — nenhuma mão detectada

print(f"📄 CSV de saída: {CSV_PATH}")

📄 CSV de saída: dataset_gestos.csv


## 🤖 5. Inicialização do Recognizer

In [5]:
base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)

options = mp_vision.GestureRecognizerOptions(
    base_options=base_options,
    running_mode=mp_vision.RunningMode.IMAGE,
    num_hands=MAX_HANDS,
    min_hand_detection_confidence=SCORE_THRESHOLD,
    min_hand_presence_confidence=SCORE_THRESHOLD,
    min_tracking_confidence=SCORE_THRESHOLD,
)

recognizer = mp_vision.GestureRecognizer.create_from_options(options)
print("✅ Recognizer pronto!")

✅ Recognizer pronto!


## 🔍 6. Funções Auxiliares

In [6]:
# Conexões dos 21 landmarks da mão
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
    (5,9),(9,13),(13,17),
]


def frame_para_mp(frame_bgr):
    """Converte frame BGR do OpenCV para MediaPipe Image."""
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)


def extrair_landmarks(hand_landmarks):
    """Extrai x, y, z dos 21 landmarks → lista de 63 valores."""
    valores = []
    for lm in hand_landmarks:
        valores.extend([lm.x, lm.y, lm.z])
    return valores  # 21 pontos × 3 coords = 63 features


def normalizar_landmarks(landmarks_raw):
    """
    Normaliza os landmarks em relação ao ponto 0 (pulso).
    Torna o dataset invariante à posição da mão na tela.
    """
    valores = np.array(landmarks_raw).reshape(21, 3)
    origem  = valores[0]          # ponto 0 = pulso
    valores = valores - origem    # centraliza no pulso
    escala  = np.max(np.abs(valores)) or 1.0
    valores = valores / escala    # normaliza escala
    return valores.flatten().tolist()


def desenhar_landmarks(frame, hand_landmarks_list, cor):
    """Desenha os 21 landmarks da mão com a cor indicada."""
    h, w = frame.shape[:2]
    for hand_landmarks in hand_landmarks_list:
        pontos = [(int(lm.x * w), int(lm.y * h)) for lm in hand_landmarks]
        for inicio, fim in HAND_CONNECTIONS:
            cv2.line(frame, pontos[inicio], pontos[fim], cor, 2)
        for ponto in pontos:
            cv2.circle(frame, ponto, 5, cor, -1)
            cv2.circle(frame, ponto, 5, (255, 255, 255), 1)
    return frame


def salvar_linha(landmarks_norm, label, csv_path):
    """Salva uma linha no CSV (cria o arquivo se não existir)."""
    colunas = [f"{eixo}{i}" for i in range(21) for eixo in ("x", "y", "z")]
    linha   = pd.DataFrame([landmarks_norm + [label]], columns=colunas + ["label"])
    escrever_header = not os.path.exists(csv_path)
    linha.to_csv(csv_path, mode="a", header=escrever_header, index=False)


print("Funções carregadas.")

Funções carregadas.


## 🎥 7. Loop de Coleta

| Ação | Como fazer |
|------|------------|
| Trocar gesto | Digite o novo nome no terminal e pressione **ENTER** |
| Encerrar | Pressione **Q** na janela da câmera |

In [9]:
import threading

estado = {"label": "", "encerrar": False}
total_gravados = 0

cap = cv2.VideoCapture(CAMERA_INDEX)

if not cap.isOpened():
    print(f"❌ Não foi possível abrir a câmera (índice {CAMERA_INDEX}).")
else:
    estado["label"] = input("Digite o nome do gesto inicial (LABEL): ").strip()

    print(f"\n✅ Gravando '{estado['label']}' continuamente.")
    print("   Para trocar o gesto: digite o novo nome aqui e pressione ENTER")
    print("   Para encerrar      : pressione Q na janela da câmera\n")

    def ler_label():
        while not estado["encerrar"]:
            novo = input()
            if novo.strip():
                estado["label"] = novo.strip()
                print(f"🔄 Label trocada para: '{estado['label']}'")

    thread = threading.Thread(target=ler_label, daemon=True)
    thread.start()

    while True:
        ret, frame = cap.read()
        if not ret:
            print("⚠️  Falha ao capturar frame.")
            break

        frame     = cv2.flip(frame, 1)
        mp_image  = frame_para_mp(frame)
        resultado = recognizer.recognize(mp_image)
        tem_mao   = bool(resultado.hand_landmarks)

        if tem_mao:
            frame = desenhar_landmarks(frame, resultado.hand_landmarks, COR_GRAVANDO)

            landmarks_raw  = extrair_landmarks(resultado.hand_landmarks[0])
            landmarks_norm = normalizar_landmarks(landmarks_raw)
            salvar_linha(landmarks_norm, estado["label"], CSV_PATH)
            total_gravados += 1

        # ── HUD ──────────────────────────────────────────────────────────────
        h, w = frame.shape[:2]

        overlay = frame.copy()
        cv2.rectangle(overlay, (0, h - 70), (w, h), (20, 20, 20), -1)
        cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

        cv2.putText(frame, f"Label: {estado['label']}",
                    (10, h - 42), cv2.FONT_HERSHEY_SIMPLEX,
                    0.65, (255, 220, 0), 2, cv2.LINE_AA)

        status     = f"Gravados: {total_gravados}  |  {'GRAVANDO' if tem_mao else 'Sem mao'}"
        cor_status = COR_GRAVANDO if tem_mao else COR_ERRO
        cv2.putText(frame, status,
                    (10, h - 14), cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, cor_status, 1, cv2.LINE_AA)

        cv2.imshow("Coletor — troque o gesto pelo terminal | Q para sair", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    estado["encerrar"] = True
    cap.release()
    cv2.destroyAllWindows()
    print(f"\n✅ Sessão encerrada.")
    print(f"   Frames gravados nesta sessão : {total_gravados}")
    print(f"   CSV salvo em                 : {CSV_PATH}")


✅ Gravando 'coração' continuamente.
   Para trocar o gesto: digite o novo nome aqui e pressione ENTER
   Para encerrar      : pressione Q na janela da câmera


✅ Sessão encerrada.
   Frames gravados nesta sessão : 155
   CSV salvo em                 : dataset_gestos.csv


## 🔎 8. Inspecionar o CSV gerado

In [ ]:
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print(f"Total de amostras : {len(df)}")
    print(f"Features          : {len(df.columns) - 1}  (21 landmarks × x/y/z)")
    print(f"\nDistribuição por label:")
    print(df["label"].value_counts().to_string())
    print(f"\nPrimeiras linhas:")
    df.head()
else:
    print(f"⚠️  Arquivo '{CSV_PATH}' ainda não existe. Execute o loop de coleta primeiro.")